In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/Colab Notebooks/RAG with Excel and PDF Data with Langchain and OpenAI

In [ ]:
from google.colab import userdata
api_key = userdata.get('Groq')

# **EXCEL**


In [ ]:
!pip install langchain-community unstructured langchain-groq faiss-cpu sentence-transformers -q

In [ ]:
#Import libraries
from langchain_community.document_loaders import UnstructuredExcelLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores.faiss import FAISS
from IPython.display import display, Markdown

In [ ]:
loader = UnstructuredExcelLoader("Sample of Employee Database.xlsx")
data = loader.load()
data[:5]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
chunks = text_splitter.split_documents(data)
chunks[0]

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db_faiss = FAISS.from_documents(chunks, embeddings)

In [ ]:
query = "Retrieve the marketing employees"
output_retrieval = db_faiss.similarity_search(query, k=3)
output_retrieval_merged = "\n\n".join([doc.page_content for doc in output_retrieval])
display(Markdown(output_retrieval_merged))

In [ ]:
prompt = f"""
based on this context {output_retrieval_merged}
answer the following question: {query}
if you don't have information on the answer, say you don't know
"""

In [ ]:
model = ChatGroq(groq_api_key=api_key,
                   model = "llama-3.3-70b-versatile",
                   temperature=0)
response_text= model.invoke(prompt)

In [ ]:
display(Markdown(response_text.content))

In [ ]:
#Functions for Excel
def prepare_excel(file_path):
    loader = UnstructuredExcelLoader(file_path)
    data = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=1000)
    chunks = text_splitter.split_documents(data)

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db_faiss = FAISS.from_documents(chunks, embeddings)

    return db_faiss


In [ ]:
def rag(query, db_faiss, k= 3):
    output_retrieval = db_faiss.similarity_search(query, k= k)

    output_retrieval_merged = "\n\n".join([doc.page_content for doc in output_retrieval])

    prompt = f"""
    based on this context {output_retrieval_merged}
    answer the following question: {query}
    if you don't have information on the answer, say you don't know
    """

    model = ChatGroq(groq_api_key=api_key,
                   model = "llama-3.3-70b-versatile",
                   temperature=0)
    response_text= model.invoke(prompt)

    return display(Markdown(response_text.content))


In [ ]:
db_excel = prepare_excel("Sample of Employee Database.xlsx")

In [ ]:
rag("Who has a car and is living in madinaty", db_excel)